In [6]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Cài đặt các thư viện cần thiết
# Sử dụng thư viện langchain-postgres mới nhất được tối ưu cho pgvector
!pip install -qU langchain langchain-openai langchain-community langchain-postgres psycopg2-binary pypdf tiktoken langchain-text-splitters ragas langchain-google-vertexai

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.7 MB/s eta 0:00:00


In [4]:
import os
from google.colab import userdata

# Lấy các key từ Colab Secrets (hoặc bạn có thể gán thẳng chuỗi string vào đây nếu làm nháp)
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')



In [10]:
!pip uninstall -y ragas langchain langchain-openai langchain-core langchain-community
!pip install -U ragas langchain-openai langchain

Found existing installation: ragas 0.3.9
Uninstalling ragas-0.3.9:
  Successfully uninstalled ragas-0.3.9
Found existing installation: langchain 0.2.17
Uninstalling langchain-0.2.17:
  Successfully uninstalled langchain-0.2.17
Found existing installation: langchain-openai 0.1.25
Uninstalling langchain-openai-0.1.25:
  Successfully uninstalled langchain-openai-0.1.25
Found existing installation: langchain-core 0.2.43
Uninstalling langchain-core-0.2.43:
  Successfully uninstalled langchain-core-0.2.43
Found existing installation: langchain-community 0.2.19
Uninstalling langchain-community-0.2.19:
  Successfully uninstalled langchain-community-0.2.19
  Using cached ragas-0.4.3-py3-none-any.whl.metadata (23 kB)
  Using cached langchain_openai-1.3.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain-1.3.8-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_core-1.4.6-py3-none-any.whl.metadata (4.5 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 k

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-1439' coro=<Executor.wrap_callable_with_index.<locals>.wrapped_callable_async() done, defined at /usr/local/lib/python3.12/dist-packages/ragas/executor.py:67> exception=ValueError('No relationships match the provided condition. Cannot form clusters.')>
Traceback (most recent call last):
  File "/usr/lib/python3.12/asyncio/tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ragas/executor.py", line 73, in wrapped_callable_async
    raise e
  File "/usr/local/lib/python3.12/dist-packages/ragas/executor.py", line 69, in wrapped_callable_async
    result = await callable(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ragas/testset/synthesizers/base.py", line 108, in generate_scenarios
    )
      
  File "/usr/local/lib/python3.12/dist-packa

In [3]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 18.2 MB/s eta 0:00:00


In [3]:
!pip install "langchain-community<0.4.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 15.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.4.2
    Uninstalling langchain-community-0.4.2:
      Successfully uninstalled langchain-community-0.4.2


In [8]:
import json
import os
from pathlib import Path

# Thêm OpenAI Client
from openai import OpenAI

from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
)
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings as RagasOpenAIEmbeddings

from langchain_core.documents import Document

# ==========================================
# CONFIG
# ==========================================

MARKDOWN_FOLDER = "/content/drive/MyDrive/thesis/enterprise-data/processed"
OUTPUT_JSON = "/content/drive/MyDrive/thesis/enterprise-data/evaluation-rag/ground-truth-data/benchmark.json"

TESTSET_SIZE = 200
OPENAI_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ĐẢM BẢO BẠN ĐÃ SET API KEY VÀO BIẾN MÔI TRƯỜNG
# os.environ["OPENAI_API_KEY"] = "sk-..."

# 1. KHỞI TẠO CLIENT OPENAI
openai_client = OpenAI()

# ==========================================
# LOAD MARKDOWN FILES
# ==========================================

documents = []

for md_file in Path(MARKDOWN_FOLDER).rglob("*.md"):
    content = md_file.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    documents.append(
        Document(
            page_content=content,
            metadata={"source": str(md_file)}
        )
    )

print(f"Loaded {len(documents)} markdown files")

# ==========================================
# NATIVE RAGAS LLM & EMBEDDING
# ==========================================

# 2. TRUYỀN CLIENT VÀO CẢ LLM VÀ EMBEDDINGS
generator_llm = llm_factory(OPENAI_MODEL, client=openai_client)
embeddings = RagasOpenAIEmbeddings(model=EMBEDDING_MODEL, client=openai_client)

# ==========================================
# RAGAS GENERATOR
# ==========================================

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=embeddings,
)

# Chạy tạo testset
testset = generator.generate_with_langchain_docs(
    documents,
    testset_size=TESTSET_SIZE,
    query_distribution=[
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1.0),
    ],
)

# ==========================================
# EXPORT JSON
# ==========================================

dataset = testset.to_pandas()

records = []
for _, row in dataset.iterrows():
    records.append(
        {
            "question": row["user_input"],
            "ground_truth_answer": row["reference"],
            "contexts": row["reference_contexts"],
        }
    )

# Đảm bảo folder đã tồn tại
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Saved {len(records)} questions")
print(f"Output: {OUTPUT_JSON}")

Loaded 1 markdown files


Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/46 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/200 [00:00<?, ?it/s]

Saved 200 questions
Output: /content/drive/MyDrive/thesis/enterprise-data/evaluation-rag/ground-truth-data/benchmark.json
